In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from collections import Counter
import gc
import time
import psutil
import os

start_time = time.time()
process = psutil.Process(os.getpid())

def print_memory_usage(step_name):
    mem_info = process.memory_info()
    mem_gb = mem_info.rss / (1024 ** 3)

    print(f"{step_name} - Memory Usage: {mem_gb:.2f} GB")
    return mem_gb

def print_step_header(step_num, step_name):
    separator = "=" * 60

    print("\n" + separator)
    print(f"STEP {step_num}: {step_name}")
    print(separator)

initial_mem = print_memory_usage("Initial")

print(f"Start Time: {time.strftime('%H:%M:%S')}")
print(f"Initial Memory: {initial_mem:.2f} GB")

Initial - Memory Usage: 0.46 GB

STARTING PROTEIN GO TERM PROCESSING
Start Time: 21:56:10
Initial Memory: 0.46 GB


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk

save_dir = "C:/Users/USER/Documents/cod3astro/ML_AI/ProteinSeq_DL/data/processed"

train_dataset_arrow = load_from_disk(f"{save_dir}/train")
val_dataset_arrow   = load_from_disk(f"{save_dir}/val")
test_dataset_arrow  = load_from_disk(f"{save_dir}/test")

print(f"Training samples:   {len(train_dataset_arrow):,}")
print(f"Validation samples: {len(val_dataset_arrow):,}")
print(f"Test samples:       {len(test_dataset_arrow):,}")

class ProteinDataset(Dataset):

    def __init__(self, hf_dataset, max_seq_length=1024):
        self.dataset = hf_dataset
        self.max_seq_length = max_seq_length

        self.label_cols = [
            col for col in hf_dataset.column_names
            if col.startswith("label_")
        ]

        self.amino_acids = "ACDEFGHIKLMNPQRSTVWY"
        self.aa_to_idx = {aa: i+1 for i, aa in enumerate(self.amino_acids)}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]

        sequence = row["sequence"]
        encoded = [self.aa_to_idx.get(aa, 0) for aa in sequence]

        encoded = encoded[:self.max_seq_length]
        if len(encoded) < self.max_seq_length:
            encoded += [0] * (self.max_seq_length - len(encoded))

        labels = [row[col] for col in self.label_cols]

        return {
            "sequence": torch.tensor(encoded, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.float32)
        }

train_dataset = ProteinDataset(train_dataset_arrow)
val_dataset   = ProteinDataset(val_dataset_arrow)
test_dataset  = ProteinDataset(test_dataset_arrow)

num_classes = len(train_dataset.label_cols)
print(f"Number of GO labels: {num_classes}")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

batch = next(iter(train_loader))

print("\nSanity check:")
print("Sequence type:", type(batch["sequence"]))
print("Sequence shape:", batch["sequence"].shape)
print("Labels shape:", batch["labels"].shape)

Training samples:   71,925
Validation samples: 15,412
Test samples:       15,413
Number of GO labels: 2761

Sanity check:
Sequence type: <class 'torch.Tensor'>
Sequence shape: torch.Size([8, 1024])
Labels shape: torch.Size([8, 2761])


In [16]:
import torch.nn as nn 
import torch.optim as optim 
from sklearn.metrics import f1_score 

class SimpleProteinCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=21,
            embedding_dim=128,
            padding_idx=0
        )

        self.conv_layers = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.ReLU(),

            nn.Conv1d(64, 32, kernel_size=7, padding=3),
            nn.ReLU()
        )

        self.global_pool = nn.AdaptiveMaxPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.conv_layers(x)
        x = self.global_pool(x)
        x = x.squeeze(-1)

        logits = self.classifier(x)
        return logits

print("✓ SimpleProteinCNN class created")

sample_model = SimpleProteinCNN(num_classes=len(train_dataset.label_cols))
total_params = sum(p.numel() for p in sample_model.parameters())

print(f"  Model parameters: {total_params:,}")
print_memory_usage("After creating model")

✓ SimpleProteinCNN class created
  Model parameters: 299,305
After creating model - Memory Usage: 0.51 GB


0.5105056762695312

In [17]:
print(type(train_dataset))

<class '__main__.ProteinDataset'>


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gc

print("Step 1: Setting up device...")
device = torch.device("cpu")
print(f"✓ Using device: {device}")

print("\nStep 2: Verifying dataset structure...")
print(f"Training proteins: {len(train_dataset):,}")
print(f"Validation proteins: {len(val_dataset):,}")
print(f"Test proteins: {len(test_dataset):,}")

# label_cols = [col for col in train_dataset.column_names if col.startswith("label_")]
if len(label_cols) == 0:
    raise ValueError("No GO term label columns found!")

num_classes = len(label_cols)
print(f"✓ Found {num_classes} GO term labels")

batch_size = 8  # small batch size for CPU

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"✓ DataLoaders created")
print(f"  Batch size: {batch_size}")
print(f"  Training batches: {len(train_loader):,}")

class LightweightProteinCNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=21, embedding_dim=32, padding_idx=0)
        self.conv = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):

        if x.dim() == 1:
            x = x.unsqueeze(0)

        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv(x))
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)

        return x

model = LightweightProteinCNN(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model created: {model.__class__.__name__}")
print(f"  Total parameters: {total_params:,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0)

print("✓ Loss function and optimizer initialized")
print("\nStep 7: Running sanity check...")

batch = next(iter(train_loader))
sequences = batch["sequence"].to(device)
labels = batch["labels"].to(device)

print(f"Batch shapes:")
print(f"  Sequences: {sequences.shape}")
print(f"  Labels: {labels.shape}")

outputs = model(sequences)
print(f"  Model outputs: {outputs.shape}")

loss = criterion(outputs, labels)
print(f"  Loss: {loss.item():.4f}")

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("✓ Forward and backward pass successful")

gc.collect()

print("\n" + "="*60)
print("FINAL TRAINING SETUP SUMMARY")
print("="*60)

print(f"Device: {device}")
print(f"Model: {model.__class__.__name__}")
print(f"Parameters: {total_params:,}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Batch size: {batch_size}")
print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

print("\nREADY FOR TRAINING")
print("="*60)

Step 1: Setting up device...
✓ Using device: cpu

Step 2: Verifying dataset structure...
Training proteins: 71,925
Validation proteins: 15,412
Test proteins: 15,413


AttributeError: 'ProteinDataset' object has no attribute 'column_names'